<a href="https://colab.research.google.com/github/Sagaustus/adh-group-projects/blob/main/group-03-african-artworks/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Complete About Possession, Silent About Transfer

### Provenance as structural absence in museum linked data

**Group 3 · working chapter draft**

---

This notebook runs the analysis end to end. Cells marked **YOUR DECISION** are where
your judgement enters.

**The thesis you are testing.** This dataset always knows where an object is now. It
never knows how it got there. That asymmetry is not an oversight in the data — it is
the shape of a schema built by and for holding institutions, and it is precisely the
information a restitution claim requires.

In [ ]:
# Setup — run this first. Nothing to upload.
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

URL = "https://raw.githubusercontent.com/Sagaustus/adh-dh-datasets/main/datasets/10_african_artworks_in_museums/data.csv"
df = pd.read_csv(URL)
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(df.dtypes.to_string())

## Step 1 · Frame

| | Question | Method |
|---|---|---|
| **Descriptive** | How concentrated is custody of African objects, and which institutions hold them? | Concentration measures and a Lorenz curve |
| **Analytical** | Do objects catalogued under colonial-era polity names differ in where they are held from those under modern state names? | Cross-tabulation, chi-square, effect size |

And one question the dataset answers by refusing to: **how did any of these objects
enter these collections?**

## Step 2 · Absence audit

Start here rather than at the description, because for this chapter the absence *is*
the finding.

In [ ]:
print("WHAT THIS SCHEMA RECORDS, AND WHAT IT DOES NOT\n")
print(f"{'field':<22}{'present':>10}")
print("-" * 34)
for c in df.columns:
    print(f"{c:<22}{df[c].notna().mean():>9.1%}")
print()
print("Now the fields that do not exist at all:\n")
for absent in ["date of acquisition", "mode of acquisition (gift/purchase/seizure)",
               "prior owner", "collector or expedition", "source community",
               "the object's own name in its language", "standing restitution claim"]:
    print(f"   MISSING ENTIRELY   {absent}")
print()
print("Read the two lists together. `held_by` is complete to the last row. Every")
print("field that would describe the transfer is not merely empty — there is no")
print("column for it. An empty field is a gap; a missing field is a decision.")

**YOUR DECISION.** Which absence most limits your questions? Two sentences.

Note the difference this dataset makes vivid: a field that is 90% empty still tells
you what the schema thought worth recording. A field that does not exist tells you
what it did not.

## Step 3 · Describe

Custody, and how unevenly it is distributed.

In [ ]:
holders = df["held_by"].value_counts()
share = holders / len(df)

print(f"{len(holders)} institutions hold {len(df):,} object records "
      f"({df['wikidata_id'].nunique():,} distinct objects)\n")
print("The ten largest holders:")
for name, n in holders.head(10).items():
    print(f"   {n:>5}  ({n/len(df):>5.1%})  {str(name)[:64]}")
print()

cum = share.sort_values(ascending=False).cumsum()
hhi = float((share ** 2).sum())
print(f"top 1 institution   {share.max():.1%}")
print(f"top 10              {cum.iloc[9]:.1%}")
print(f"top 25              {cum.iloc[24]:.1%}")
print(f"Herfindahl-Hirschman index  {hhi:.4f}")
print()
print("HHI is borrowed from competition economics: 0 is perfectly dispersed, 1 is a")
print("single holder. Use it because it gives a reader one comparable number, and")
print("say where you borrowed it from.")

In [ ]:
origins = df["country_of_origin"].value_counts()
print(f"{len(origins)} distinct values in country_of_origin\n")
print(origins.head(12).to_string())
print()
print("Look carefully at that list before treating it as a country column.")

### The column that is not a country column

`country_of_origin` mixes at least three incompatible things:

- **Modern states** — Democratic Republic of the Congo, Nigeria
- **Historical polities** — Ottoman Empire, Belgian Congo, Dahomey
- **Chronological periods used as places** — Ancient Egypt

A flat field is carrying two different theories of political time, and the schema
treats them identically. Quantify it.

In [ ]:
HISTORICAL_POLITIES = [
    "Ancient Egypt", "Ottoman Empire", "Belgian Congo", "Dahomey", "Zaire",
    "Rhodesia", "Gold Coast", "French West Africa", "Tanganyika", "Abyssinia",
    "Nyasaland", "Bechuanaland", "South West Africa", "Upper Volta",
]
found = {c: int((df["country_of_origin"] == c).sum())
         for c in HISTORICAL_POLITIES if (df["country_of_origin"] == c).any()}
df["historical_name"] = df["country_of_origin"].isin(found.keys())

print("Historical polity names present in the data:")
for c, n in sorted(found.items(), key=lambda kv: -kv[1]):
    print(f"   {n:>5}  {c}")
print()
print(f"{df['historical_name'].sum():,} of {len(df):,} records "
      f"({df['historical_name'].mean():.0%}) are catalogued under a polity that no")
print("longer exists. 'Belgian Congo' is not a historical note here — it is a live")
print("value in a current linked-data field.")
print()
print("YOUR DECISION: is 'Ancient Egypt' the same kind of value as 'Belgian Congo'?")
print("Your answer changes the number above, so argue for it in the chapter.")

## Step 4 · Compare

Two groups that matter: objects catalogued under a historical polity name against
those under a modern state name. Does the naming travel with anything?

In [ ]:
hist = df[df["historical_name"]]
modern = df[~df["historical_name"]]

print(f"{'':<34}{'historical':>12}{'modern':>10}")
print("-" * 58)
print(f"{'records':<34}{len(hist):>12,}{len(modern):>10,}")
print(f"{'distinct holding institutions':<34}{hist['held_by'].nunique():>12}{modern['held_by'].nunique():>10}")
print(f"{'share held by the top institution':<34}"
      f"{hist['held_by'].value_counts().iloc[0]/len(hist):>11.1%}"
      f"{modern['held_by'].value_counts().iloc[0]/len(modern):>10.1%}")
print(f"{'inception year recorded':<34}"
      f"{hist['inception_year'].notna().mean():>11.1%}"
      f"{modern['inception_year'].notna().mean():>10.1%}")
print()
print("Top holders of HISTORICAL-name objects:")
for name, n in hist["held_by"].value_counts().head(5).items():
    print(f"   {n:>5}  {str(name)[:60]}")
print()
print("Top holders of MODERN-name objects:")
for name, n in modern["held_by"].value_counts().head(5).items():
    print(f"   {n:>5}  {str(name)[:60]}")

## Step 5 · Test

A test **and** an effect size. Then the finding this dataset hides in its duplicate
rows.

In [ ]:
from scipy.stats import chi2_contingency

TOP_N = 12
top_holders = df["held_by"].value_counts().head(TOP_N).index
sub = df[df["held_by"].isin(top_holders)]

table = pd.crosstab(sub["held_by"], sub["historical_name"])
chi2, p, dof, expected = chi2_contingency(table)
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

print(f"chi-square {chi2:,.1f}   dof {dof}   p = {p:.2e}")
print(f"Cramer's V (effect size) = {cramers_v:.3f}")
print()
print("Cramer's V rather than phi, because this is larger than 2x2.")
print("Conventional reading: 0.1 small, 0.3 medium, 0.5 large.")
print()
print("Holding institutions are not interchangeable with respect to how their")
print("African holdings are catalogued. Which is a claim about cataloguing")
print("practice, and not yet a claim about anything else.")

### The duplicate rows are not an error

1,928 rows carry only 1,627 distinct identifiers. The obvious reading is a data
quality problem. Look at what the repeats actually are.

In [ ]:
dups = df[df["wikidata_id"].duplicated(keep=False)]
by_id = dups.groupby("wikidata_id").agg(rows=("held_by", "size"),
                                        institutions=("held_by", "nunique"))
multi = by_id[by_id["institutions"] > 1]

print(f"{len(dups):,} rows carry a repeated identifier")
print(f"{len(by_id)} distinct identifiers repeat")
print(f"{len(multi)} of those are held by MORE THAN ONE institution")
print()
print("These are not duplicates. They are single objects distributed across")
print("several collections — and the data model can only express that as the same")
print("identifier appearing several times.\n")

for wid in multi.sort_values("institutions", ascending=False).head(3).index:
    rows = df[df["wikidata_id"] == wid]
    print(f"   {rows['object'].iloc[0][:56]}  ({rows['country_of_origin'].iloc[0]})")
    for h in rows["held_by"]:
        print(f"      held by: {str(h)[:66]}")
    print()

**Look at what that shows.** A single object — a manuscript, a set of folios, a
dismembered assemblage — divided between institutions, sometimes including one in the
country of origin.

This is the restitution question in its most concrete form, and the schema can only
say it by accident: as a repeated identifier. There is no field for *dispersal*, no
field for *reassembly*, and no field for the claim.

**This is your chapter's strongest single piece of evidence.** Step 7 codes it.

## Step 6 · Show

One chart. Concentration is the descriptive finding, so plot the Lorenz curve — it
shows the whole distribution rather than a top-ten league table, and it lets a reader
see how steep the inequality is.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5))

# Lorenz curve of custody
counts_sorted = np.sort(holders.values)
cum_objects = np.cumsum(counts_sorted) / counts_sorted.sum()
cum_inst = np.arange(1, len(counts_sorted) + 1) / len(counts_sorted)

# Gini from the standard formula rather than an integration helper: numpy renamed
# trapz to trapezoid in 2.0, and a notebook students run on an unknown numpy
# should not depend on which one they have.
k = len(counts_sorted)
idx = np.arange(1, k + 1)
gini = float((2 * (idx * counts_sorted).sum()) / (k * counts_sorted.sum()) - (k + 1) / k)

ax1.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="perfect dispersal")
ax1.plot(cum_inst, cum_objects, lw=2, color="#9c2c1f",
         label=f"observed (Gini = {gini:.2f})")
ax1.fill_between(cum_inst, cum_objects, cum_inst, alpha=.12, color="#9c2c1f")
ax1.set_xlabel("cumulative share of institutions")
ax1.set_ylabel("cumulative share of object records")
ax1.set_title("Custody is concentrated")
ax1.legend(loc="upper left")

# The ten largest holders
top10 = holders.head(10)
labels = [str(s)[:34] + ("…" if len(str(s)) > 34 else "") for s in top10.index]
ax2.barh(labels[::-1], top10.values[::-1], color="#2b5c50")
ax2.set_xlabel("object records held")
ax2.set_title("The ten largest holders")
ax2.tick_params(labelsize=8)

plt.tight_layout()
print(f"CAPTION. Concentration of custody across {len(holders)} institutions holding")
print(f"{len(df):,} object records of African origin (Wikidata, no filtering applied).")
print(f"Left: Lorenz curve; the {len(holders)} institutions are ordered from smallest")
print(f"to largest holding. Gini = {gini:.2f}. Right: the ten largest holders by count.")

## Step 7 · The qualitative half

Code the dispersed objects. This is a small, tractable sample with an unusually high
return: for each object held by more than one institution, establish **whether any
holder is in the country of origin**.

That single judgement turns a data artefact into evidence about where African material
heritage physically is.

In [ ]:
sample = multi.sort_values("institutions", ascending=False).head(20).index
print(f"{len(sample)} dispersed objects to code. For each, look up every holding")
print("institution and decide where it is:\n")
for i, wid in enumerate(sample, 1):
    rows = df[df["wikidata_id"] == wid]
    print(f"{i:>3}. {str(rows['object'].iloc[0])[:50]:<52}"
          f"({str(rows['country_of_origin'].iloc[0])[:22]})")
    for h in rows["held_by"].unique():
        print(f"       {str(h)[:70]}")
    print(f"       https://www.wikidata.org/wiki/{wid}")
    print()

### The coding scheme

One label per dispersed object.

| Label | Definition |
|---|---|
| `ALL_ABROAD` | Every holding institution is outside the country or region of origin |
| `SPLIT` | Holders include at least one in the country/region of origin and at least one outside |
| `ALL_IN_ORIGIN` | Every holder is in the country or region of origin |
| `UNCLEAR` | An institution cannot be located, or the origin value is a period rather than a place |

**Two coders, independently.** `UNCLEAR` will be commoner than you expect — partly
because `country_of_origin` sometimes names a period. That is a finding about the
schema, not a coding failure, and it belongs in the chapter.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from collections import Counter

# Replace with your real codes once both coders have worked through the twenty.
coder_a = ["ALL_ABROAD","SPLIT","ALL_ABROAD","UNCLEAR","ALL_ABROAD","ALL_ABROAD",
           "SPLIT","ALL_ABROAD","UNCLEAR","ALL_ABROAD","ALL_ABROAD","SPLIT",
           "UNCLEAR","ALL_ABROAD","ALL_ABROAD","SPLIT","ALL_ABROAD","UNCLEAR",
           "ALL_ABROAD","ALL_ABROAD"]
coder_b = ["ALL_ABROAD","SPLIT","ALL_ABROAD","UNCLEAR","ALL_ABROAD","SPLIT",
           "SPLIT","ALL_ABROAD","UNCLEAR","ALL_ABROAD","UNCLEAR","SPLIT",
           "UNCLEAR","ALL_ABROAD","ALL_ABROAD","SPLIT","ALL_ABROAD","ALL_ABROAD",
           "ALL_ABROAD","ALL_ABROAD"]

kappa = cohen_kappa_score(coder_a, coder_b)
raw = float(np.mean([x == y for x, y in zip(coder_a, coder_b)]))
print(f"raw agreement {raw:.2f}   Cohen's kappa {kappa:.3f}\n")
print("distribution (coder A):", dict(Counter(coder_a)))
print()
print("Disagreements:")
for i, (x, y) in enumerate(zip(coder_a, coder_b), 1):
    if x != y:
        print(f"   object {i}: {x} vs {y}")
print()
print("If ALL_ABROAD dominates, say so in exactly those terms and with this kappa.")
print("It is a quantified statement about dispersal, from a dataset with no field")
print("for dispersal — which is the methodological point of the whole chapter.")

## Step 8 · Limits

**What this analysis supports**

- Statements about this Wikidata extract, on its stated retrieval date.
- The concentration of custody at the Gini and HHI reported.
- That the schema has no field for acquisition, and that `country_of_origin` mixes
  states, historical polities and periods.
- That a specified number of objects are recorded in more than one collection.

**What it does not support**

- Any claim about *how* an object was acquired. There is no field, and inferring
  seizure from a museum's location is not evidence.
- Any claim about the totality of African objects in museums. Wikidata's coverage is
  partial and volunteer-driven; this is a sample of a sample.
- Any claim that a specific object should be restituted. That is a legal and ethical
  argument this data can inform and cannot settle.
- Any claim about institutions absent from the data. Silence here is not evidence of
  an empty collection.

**YOUR DECISION.** Add two more sentences this data does not support.

---

# Step 9 · The chapter template

Fill the gaps. Rewrite the frames in your own voice once the argument is visible.
**Length: 6,000–8,000 words.**

---

## §1 Introduction — *about 800 words*

> Debates over the restitution of African cultural heritage increasingly turn on
> ______________ . Yet the linked-data infrastructures that describe these objects
> ______________ . This chapter analyses **_____ object records** held by
> **_____ institutions**, and argues that ______________ .

*Write last.*

---

## §2 The standard and the debate — *about 1,200 words*

Wikidata's model for cultural objects — name the properties. Then the restitution
literature: Sarr–Savoy, the Benin Dialogue Group, national claims.

> Objects are described through properties including ______________ . Provenance
> can be expressed through ______________ , but is ______________ in practice.
> The Sarr–Savoy report requires ______________ , which ______________ .

---

## §3 Data — *about 900 words*

> The dataset comprises **_____ records** of **_____ distinct objects**, held by
> **_____ institutions** and attributed to **_____ origin values**.
> `held_by` is present for **_____%** of records; no field records acquisition.

Then the absences as argument, from Step 2.

---

## §4 Method — *about 700 words*

> Custody concentration was measured with ______________ and ______________ .
> Records were classified as carrying a historical or a modern polity name using
> ______________ ; this list is reported in full because ______________ .
> Objects appearing under a single identifier in more than one collection were
> identified and coded independently by two readers.

**Report the historical-polity list in an appendix.** It is a decision, and a
different list gives a different number.

---

## §5 Findings — *about 1,800 words*

**§5.1 Custody is concentrated**

> The ten largest holders account for **_____%** of records and the twenty-five
> largest for **_____%** (Gini = _____, HHI = _____).

**§5.2 The schema is complete about possession and silent about transfer**

> `held_by` is populated for **_____%** of records. No field records
> ______________ , ______________ or ______________ .

**§5.3 Colonial nomenclature persists**

> **_____ records (_____%)** are catalogued under a polity that no longer exists,
> including **_____** under "Belgian Congo" and **_____** under "Ottoman Empire".

**§5.4 Objects are dispersed, and the schema says so only by accident**

> **_____** objects appear in more than one collection. Coded independently
> (κ = _____), **_____** of twenty were held entirely outside their region of
> origin.

**Figure 1** after §5.1, captioned from Step 6.

---

## §6 Discussion — *about 1,400 words*

> The asymmetry between recorded custody and unrecorded acquisition suggests
> ______________ . For restitution claims, the consequence is ______________ .

The counter-practice, required:

> Initiatives such as ______________ respond by ______________ .

Candidates: Local Contexts and Traditional Knowledge Labels, the Benin Dialogue
Group, Digital Benin, the German *Kontaktzone* provenance projects.

---

## §7 Limitations · §8 Conclusion · §9 References · §10 Data and code

In [ ]:
print("=" * 74)
print("DRAFT SENTENCES — your numbers already placed")
print("=" * 74)

print("""
§3 DATA
  The dataset comprises {r:,} records describing {o:,} distinct objects of African
  origin, held by {i} institutions and attributed to {c} distinct origin values.
  The holding institution is recorded for {h:.0%} of records. Date of creation is
  present for {y:.0%}. No field in the schema records the date or mode of
  acquisition, the prior owner, or any collector or expedition.
""".format(r=len(df), o=df["wikidata_id"].nunique(), i=df["held_by"].nunique(),
           c=df["country_of_origin"].nunique(), h=df["held_by"].notna().mean(),
           y=df["inception_year"].notna().mean()))

print("""§5.1 CUSTODY IS CONCENTRATED
  The largest single holder accounts for {t1:.1%} of records, the ten largest for
  {t10:.1%} and the twenty-five largest for {t25:.1%} (Gini = {g:.2f},
  Herfindahl-Hirschman index = {hhi:.4f}). The five largest holdings are held by
  institutions in Europe and North America.
""".format(t1=share.max(), t10=cum.iloc[9], t25=cum.iloc[24], g=gini, hhi=hhi))

print("""§5.3 COLONIAL NOMENCLATURE PERSISTS
  {n:,} records ({p:.0%}) are catalogued under a polity that no longer exists.
  These include {bc} records attributed to the Belgian Congo and {ot} to the
  Ottoman Empire, alongside {ae} attributed to Ancient Egypt - a chronological
  designation occupying the same field as a modern state.
""".format(n=int(df["historical_name"].sum()), p=df["historical_name"].mean(),
           bc=int((df["country_of_origin"]=="Belgian Congo").sum()),
           ot=int((df["country_of_origin"]=="Ottoman Empire").sum()),
           ae=int((df["country_of_origin"]=="Ancient Egypt").sum())))

print("""§5.4 OBJECTS ARE DISPERSED
  {m} objects appear under a single identifier in more than one collection,
  accounting for {d:,} rows. The schema has no field for dispersal; the fact is
  expressible only as a repeated identifier. Twenty such objects were coded
  independently by two readers (kappa = {k:.3f}).
""".format(m=len(multi), d=len(dups), k=kappa))
print("=" * 74)
print("Left for you: what it MEANS, what it cannot support, the counter-practice.")

### Three mistakes that sink first chapters

**Inferring acquisition from location.** A Belgian museum holding a Congolese object
is suggestive and is not evidence of how it was acquired. The chapter's power comes
from naming the absence, not from filling it with assumption.

**Treating the duplicate rows as dirty data.** They are the most interesting thing in
the file. A chapter that deduplicates in a cleaning step and never mentions it has
thrown away its best evidence.

**Letting Wikidata stand for museums.** This is volunteer-curated, partial, and
skewed toward institutions that publish open data. Say so in §3, not only in §7.

---

# Step 10 · Dividing the work

More than five people, one chapter. Divide by **expertise**, not by paragraph count.

## The roles

| # | Role | Owns | Expertise it draws on | Hands over |
|---|---|---|---|---|
| 1 | **Corpus &amp; provenance** | §3, absence audit | Museum documentation, collections management | The schema audit and the historical-polity list |
| 2 | **Analysis** | §4, §5.1–5.3 | Statistics, computation | Gini, HHI, the chi-square, the dispersal count |
| 3 | **Coder A** | §5.4, jointly | Art history, geography of collections | 20 independently assigned codes |
| 4 | **Coder B** | §5.4, jointly | Art history, geography of collections | 20 independently assigned codes |
| 5 | **Theory &amp; restitution** | §2, §6 | Postcolonial studies, heritage law | Sarr–Savoy, the claims literature, the counter-practice |
| 6 | **Visualisation &amp; communication** | Figure 1, §8 | Design, editing | A Lorenz curve that argues unaided |
| 7 | **Integration editor** | §1, §7, references | Editorial judgement | One voice, and a chapter that ends |

Role 5 carries more weight here than in the other groups: the restitution literature
is large, contested and legally technical, and a chapter that gets Sarr–Savoy wrong
will be dismissed on that alone.

## Why coders 3 and 4 are two people

A **method requirement**. Locating twenty institutions and judging whether any sits in
the region of origin involves real discretion — particularly for `UNCLEAR` cases where
the origin value is a period. Cohen's kappa is what converts two people's discretion
into a reportable figure. One coder produces an opinion.

## The order things happen in

```
   Corpus & provenance ──┐
                         ├──► Analysis ──► Visualisation ──┐
   Coders A + B ─────────┘                                 ├──► Integration
                                                           │
   Theory & restitution ───────────────────────────────────┘
```

Theory starts on day one. Integration starts last, and needs real time.

## Combining the drafts

**One person edits for voice, and everyone accepts the edit.**

**Agree the terms in writing first.** For this chapter: do you say *object*, *item*
or *work*? *Holding institution* or *museum*? And critically — do you write
*restitution*, *repatriation* or *return*? They are not synonyms in the literature,
and switching between them will read as not knowing.

## Declaring who did what

Use **CRediT**. It is how the art historian and the statistician each get credited for
the thing they actually did.

In [ ]:
TEAM = {
    "Corpus & provenance":     ("________________", "data curation, investigation"),
    "Analysis":                ("________________", "formal analysis, software, methodology"),
    "Coder A":                 ("________________", "investigation, validation"),
    "Coder B":                 ("________________", "investigation, validation"),
    "Theory & restitution":    ("________________", "conceptualisation, writing - original draft"),
    "Visualisation & comms":   ("________________", "visualisation, writing - review and editing"),
    "Integration editor":      ("________________", "writing - review and editing, supervision"),
}
print("AUTHOR CONTRIBUTIONS (CRediT)")
print()
for role, (name, credit) in TEAM.items():
    print(f"  {name}: {credit}.")
    print(f"      [{role}]")
print()
print("Fill in names, delete the bracketed labels, place after the conclusion.")
print("Agree authorship order EARLY.")

### What goes wrong, and how to see it coming

**The restitution literature swallows the chapter.** It is vast and it is not your
finding. Role 5 supplies §2 and §6; the evidence is still yours.

**The coders talk.** Fatal to the kappa. If it happens, report it in §4.

**Nobody codes the UNCLEAR cases carefully.** They are the ones that reveal the
schema's confusion between place and period, and they are easy to discard as noise.

**Nobody owns the ending.** Role 7 owns §8 and owns the decision that it is finished.

## What to hand in

1. **One page**: the finding, the method, the uncertainty, and the limits list.
2. **One chart**, captioned, saying what you filtered.
3. **The coding sheet** with both coders' labels and the kappa.
4. **The historical-polity list** you used, as an appendix.

### Turning this into the chapter

Your spine is the strongest of the five groups: a schema complete about possession and
structurally silent about transfer, colonial polity names alive in a current field, and
objects dispersed across collections that the data model can only express by accident.

What it needs from you is **§6** — what follows for restitution when the infrastructure
records custody perfectly and acquisition not at all — and the counter-practice.
Digital Benin is the obvious comparison: built specifically to hold what this schema
cannot.